In [ ]:
""" frame preprocess
Convert the V-Net output 
from (64 512 512)pixel ( 65 130 130)nm 16bit
to   (32 512 512)pixel (130 130 130)nm 16bit
To save storage space.
"""

import torch
import torch.nn.functional as F
import numpy as np
import os
import tifffile
import tqdm

frames_load_fold = ""
frames_save_fold = ""

frames_list = os.listdir(frames_load_fold)
for index in tqdm.tqdm(range(len(frames_list))):
    # read 16 bit tiff, note that the order of os.listdir is not guaranteed
    frame = tifffile.imread(
        os.path.join(frames_load_fold, frames_list[index])
    ).astype(np.float32)
    frame = torch.from_numpy(frame).float()
    # 64 512 512 -> 32 512 512
    frame = F.interpolate(
        frame.unsqueeze(0).unsqueeze(0), 
        size = (32, 512, 512)
    ).squeeze(0).squeeze(0).half()
    # save 16 bit with file name formation
    tifffile.imwrite(
        "{}/{:05}.tif".format(frames_save_fold, int(frames_list[index][11:-4])),
        frame.numpy()
    )

100%|██████████| 1000/1000 [01:48<00:00,  9.20it/s]


In [ ]:
""" concatenate two or more 3D subframes into a 3D frame 
"""

import numpy as np
import tifffile
import os

LT2RD_fold = [

]
FN_fold = ""  # final
axis = 2    # 1 for vertical, 2 for horizontal

if not os.path.exists(FN_fold): os.makedirs(FN_fold)
for i in range(len(os.listdir(LT2RD_fold[0]))): 
    tifffile.imwrite(
        os.path.join(FN_fold, os.listdir(LT2RD_fold[0])[i]),
        np.concatenate([
            tifffile.imread(
                os.path.join(LT2RD_fold[j], os.listdir(LT2RD_fold[j])[i])
            ) for j in range(len(LT2RD_fold)) 
        ], axis=axis)
    )

In [ ]:
""" put subframes into whole frame
"""

import numpy as np
import tifffile
import os
import tqdm

rng_sub_user = [0, 1, 26, 27, 18, 22]

dim_label = np.array([32*4, 512*8, 512*8]).astype(int)
frame_fold = ""
whole_fold = ""

for i in tqdm.tqdm(range(len(os.listdir(frame_fold)))):
    frame_path = os.path.join(frame_fold, os.listdir(frame_fold)[i])
    whole_path = os.path.join(whole_fold, os.listdir(frame_fold)[i])
    frame = tifffile.imread(frame_path).astype(np.float32)
    if os.path.exists(whole_path):
        whole = tifffile.imread(whole_path).astype(np.float32)
    else:
        raise FileNotFoundError
        whole = np.zeros(dim_label).astype(np.float32)
    whole[
        rng_sub_user[0]*128 : rng_sub_user[1]*128,
        rng_sub_user[2]*128 : rng_sub_user[3]*128,
        rng_sub_user[4]*128 : rng_sub_user[5]*128,
    ] = frame
    if not os.path.exists(whole_fold): os.makedirs(whole_fold)
    tifffile.imwrite(whole_path, whole)